# Model_Training_01 — RepCount Baseline

This notebook trains the first baseline for repetition count prediction using prepared train/valid artifacts.


In [25]:
#!pip install torch numpy pandas 
!pip install ultralytics


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 810.4/810.4 kB 52.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.2/40.2 MB 96.9 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 75.2 MB/s eta 0:00:00

[notice] A new release of pip is available: 25.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


## 1) Objective and Evaluation
- Goal: real-time repetition count estimation from pose/video signals.
- Primary metric: **MAE** on repetition count.
- Required: **per-class MAE**.
- Optional: RMSE, within-1 accuracy, latency/FPS.

### Baseline policy for this notebook
- Start **without imbalance correction** (plain loss + plain sampling).
- Add class weighting/sampler only if per-class metrics show minority underperformance.



In [13]:
import os
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print('Libraries loaded. Seed set to', SEED)



Libraries loaded. Seed set to 42


## 2) Load Prepared Data
Use only data-preparation outputs (train/valid curated).


In [14]:
DATA_DIR = Path('../../Data/LLSP/annotation_cleaned')
TRAIN_CLEAN = DATA_DIR / 'train_cleaned.csv'
VALID_CLEAN = DATA_DIR / 'valid_cleaned.csv'
CLASS_W = DATA_DIR / 'class_weights_train.csv'
SAMPLE_W = DATA_DIR / 'train_sample_weights.csv'
MANIFEST = DATA_DIR / 'decisions_manifest.json'

train_df = pd.read_csv(TRAIN_CLEAN)
valid_df = pd.read_csv(VALID_CLEAN)
class_w_df = pd.read_csv(CLASS_W) if CLASS_W.exists() else None
sample_w_df = pd.read_csv(SAMPLE_W) if SAMPLE_W.exists() else None
manifest = json.load(open(MANIFEST)) if MANIFEST.exists() else {}

print(f'train={len(train_df)}, valid={len(valid_df)}')
print('train labels:', sorted(train_df['type'].unique()))
print('valid labels:', sorted(valid_df['type'].unique()))
print('manifest loaded:', bool(manifest))



train=639, valid=113
train labels: ['bench_pressing', 'front_raise', 'jump_jacks', 'pull_up', 'push_up', 'sit_up', 'squat']
valid labels: ['bench_pressing', 'front_raise', 'jump_jacks', 'pull_up', 'push_up', 'sit_up', 'squat']
manifest loaded: True


In [15]:
required_cols = {'name', 'type', 'count'}
missing_train = required_cols - set(train_df.columns)
missing_valid = required_cols - set(valid_df.columns)

if missing_train or missing_valid:
    raise ValueError(f'Missing required columns - train:{missing_train}, valid:{missing_valid}')

print('Required columns present.')
print('Train class distribution:')
print(train_df['type'].value_counts().to_string())



Required columns present.
Train class distribution:
type
squat             102
pull_up            94
bench_pressing     93
sit_up             93
front_raise        92
push_up            89
jump_jacks         76


## 3) Modeling Design (YOLO Pose + Temporal Counter)

### Recommended training flow
1. Extract pose keypoints per video using **YOLOv8/YOLO11 Pose**.
2. Build per-video temporal tensors: shape `[T, F]` where:
   - `T`: sequence length (frames or sampled timesteps)
   - `F`: keypoint feature dimension (e.g., flattened keypoints, angles, velocities)
3. Train a temporal regressor to predict repetition count.

### This notebook scaffolds the temporal regressor stage
- It assumes you will produce a pose-feature file per video (`.npy`/`.npz`/parquet) and map each video name to that file.



## 4) Training Config


In [16]:
CFG = {
    'run_name': 'baseline_no_imbalance',
    'epochs': 30,
    'batch_size': 16,
    'lr': 1e-3,
    'weight_decay': 1e-4,
    'num_workers': 0,
    'device': 'cuda',  # fallback to cpu in code below

    # temporal tensor shape assumptions (update after pose-feature extraction)
    'seq_len': 64,
    'feat_dim': 64,
    'hidden_dim': 128,
    'dropout': 0.2,

    # feature index (generated by pose feature extraction step)
    'feature_index_path': str(DATA_DIR / 'pose_feature_index.csv'),

    # output
    'output_root': './training_outputs',

    # baseline policy
    'use_class_weights': False,
    'use_weighted_sampler': False,
}

CFG



{'run_name': 'baseline_no_imbalance',
 'epochs': 30,
 'batch_size': 16,
 'lr': 0.001,
 'weight_decay': 0.0001,
 'num_workers': 0,
 'device': 'cuda',
 'seq_len': 64,
 'feat_dim': 64,
 'hidden_dim': 128,
 'dropout': 0.2,
 'feature_index_path': '../../Data/LLSP/annotation_cleaned/pose_feature_index.csv',
 'output_root': './training_outputs',
 'use_class_weights': False,
 'use_weighted_sampler': False}

In [17]:
# Label mapping (stable order from train)
class_names = sorted(train_df['type'].unique())
class_to_idx = {c: i for i, c in enumerate(class_names)}
idx_to_class = {i: c for c, i in class_to_idx.items()}

train_df['class_idx'] = train_df['type'].map(class_to_idx)
valid_df['class_idx'] = valid_df['type'].map(class_to_idx)

print('num_classes =', len(class_names))
print(class_to_idx)



num_classes = 7
{'bench_pressing': 0, 'front_raise': 1, 'jump_jacks': 2, 'pull_up': 3, 'push_up': 4, 'sit_up': 5, 'squat': 6}


/var/folders/7q/522tn7j14w119bz944bz8qy80000gn/T/ipykernel_82397/74867761.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_df['class_idx'] = train_df['type'].map(class_to_idx)
/var/folders/7q/522tn7j14w119bz944bz8qy80000gn/T/ipykernel_82397/74867761.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  valid_df['class_idx'] = valid_df['type'].map(class_to_idx)


## 5) PyTorch Setup and Model Skeleton


In [18]:
TORCH_AVAILABLE = True
try:
    import torch
    import torch.nn as nn
    from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
except Exception as e:
    TORCH_AVAILABLE = False
    print('PyTorch not available in this environment:', e)

if TORCH_AVAILABLE:
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    # deterministic-friendly defaults
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    device = torch.device('cuda' if (CFG['device'] == 'cuda' and torch.cuda.is_available()) else 'cpu')
    print('Using device:', device)



Using device: cpu


In [19]:
if TORCH_AVAILABLE:
    class TemporalCountRegressor(nn.Module):
        """Baseline temporal regressor: mean-pool over time + MLP head."""
        def __init__(self, feat_dim=64, hidden_dim=128, dropout=0.2):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(feat_dim, hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_dim, hidden_dim // 2),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_dim // 2, 1),
            )

        def forward(self, x):
            # x: [B, T, F]
            x = x.mean(dim=1)    # [B, F]
            y = self.net(x)      # [B, 1]
            return y.squeeze(-1) # [B]

    model = TemporalCountRegressor(
        feat_dim=CFG['feat_dim'],
        hidden_dim=CFG['hidden_dim'],
        dropout=CFG['dropout'],
    ).to(device)

    print(model.__class__.__name__)



TemporalCountRegressor


## 6) Metrics (MAE + Per-Class MAE)


In [20]:
def mae(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.mean(np.abs(y_true - y_pred)))


def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


def within_1_acc(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.mean(np.abs(y_true - y_pred) <= 1.0))


def per_class_mae(df_eval, y_pred_col='pred_count'):
    out = (
        df_eval
        .assign(abs_err=lambda d: np.abs(d['count'].astype(float) - d[y_pred_col].astype(float)))
        .groupby('type', as_index=False)['abs_err']
        .mean()
        .rename(columns={'abs_err': 'mae'})
        .sort_values('mae', ascending=False)
    )
    return out



## 7) Training Pipeline (Feature Merge, Dataset, Loop, Exports)

This section runs end-to-end training once pose features are available in `pose_feature_index.csv`.


In [21]:
from datetime import datetime, timezone

TRAIN_RESULTS = None

if not TORCH_AVAILABLE:
    print('Skip training pipeline: PyTorch is not available.')
else:
    feature_index_path = Path(CFG['feature_index_path'])
    output_dir = Path(CFG['output_root']) / CFG['run_name']
    ckpt_dir = output_dir / 'checkpoints'
    output_dir.mkdir(parents=True, exist_ok=True)
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    if not feature_index_path.exists():
        print(f"Feature index not found: {feature_index_path}")
        print('Create pose features first, then re-run this cell.')
    else:
        feat_idx = pd.read_csv(feature_index_path)

        required_feat_cols = {'name', 'feature_path'}
        missing_feat_cols = required_feat_cols - set(feat_idx.columns)
        if missing_feat_cols:
            raise ValueError(f'pose_feature_index.csv missing columns: {missing_feat_cols}')

        # normalize names for safe merge
        feat_idx = feat_idx.copy()
        feat_idx['name_norm'] = feat_idx['name'].astype(str).str.strip().str.lower()

        train_local = train_df.copy()
        valid_local = valid_df.copy()
        train_local['name_norm'] = train_local['name'].astype(str).str.strip().str.lower()
        valid_local['name_norm'] = valid_local['name'].astype(str).str.strip().str.lower()

        train_m = train_local.merge(feat_idx[['name_norm', 'feature_path']], on='name_norm', how='left')
        valid_m = valid_local.merge(feat_idx[['name_norm', 'feature_path']], on='name_norm', how='left')

        train_missing = int(train_m['feature_path'].isna().sum())
        valid_missing = int(valid_m['feature_path'].isna().sum())
        print(f'train missing features: {train_missing}')
        print(f'valid missing features: {valid_missing}')

        # drop rows with missing features for this baseline
        train_m = train_m.dropna(subset=['feature_path']).reset_index(drop=True)
        valid_m = valid_m.dropna(subset=['feature_path']).reset_index(drop=True)

        if len(train_m) == 0 or len(valid_m) == 0:
            raise RuntimeError('No train/valid rows with feature files. Cannot train baseline.')

        print(f'train rows used: {len(train_m)}, valid rows used: {len(valid_m)}')

        def _load_feature_array(path):
            path = Path(path)
            if not path.exists():
                raise FileNotFoundError(f'Feature file not found: {path}')

            if path.suffix.lower() == '.npy':
                arr = np.load(path)
            elif path.suffix.lower() == '.npz':
                z = np.load(path)
                if 'x' in z:
                    arr = z['x']
                elif 'features' in z:
                    arr = z['features']
                else:
                    arr = z[z.files[0]]
            else:
                raise ValueError(f'Unsupported feature file extension: {path.suffix}')

            arr = np.asarray(arr, dtype=np.float32)
            if arr.ndim == 1:
                arr = arr[:, None]
            if arr.ndim != 2:
                raise ValueError(f'Expected 2D feature array [T,F], got shape={arr.shape} for {path}')
            return arr

        def _fix_shape(arr, seq_len, feat_dim):
            # adjust T
            if arr.shape[0] >= seq_len:
                arr = arr[:seq_len, :]
            else:
                pad_t = np.zeros((seq_len - arr.shape[0], arr.shape[1]), dtype=np.float32)
                arr = np.concatenate([arr, pad_t], axis=0)

            # adjust F
            if arr.shape[1] >= feat_dim:
                arr = arr[:, :feat_dim]
            else:
                pad_f = np.zeros((arr.shape[0], feat_dim - arr.shape[1]), dtype=np.float32)
                arr = np.concatenate([arr, pad_f], axis=1)
            return arr

        class RepCountFeatureDataset(Dataset):
            def __init__(self, df, seq_len, feat_dim):
                self.df = df.reset_index(drop=True)
                self.seq_len = seq_len
                self.feat_dim = feat_dim

            def __len__(self):
                return len(self.df)

            def __getitem__(self, idx):
                row = self.df.iloc[idx]
                arr = _load_feature_array(row['feature_path'])
                arr = _fix_shape(arr, self.seq_len, self.feat_dim)

                x = torch.tensor(arr, dtype=torch.float32)
                y = torch.tensor(float(row['count']), dtype=torch.float32)
                c = torch.tensor(int(row['class_idx']), dtype=torch.long)

                meta = {
                    'name': str(row['name']),
                    'type': str(row['type'])
                }
                return x, y, c, meta

        train_ds = RepCountFeatureDataset(train_m, CFG['seq_len'], CFG['feat_dim'])
        valid_ds = RepCountFeatureDataset(valid_m, CFG['seq_len'], CFG['feat_dim'])

        # optional weighted sampler
        sampler = None
        if CFG['use_weighted_sampler'] and sample_w_df is not None:
            weight_col_candidates = [c for c in sample_w_df.columns if 'weight' in c.lower()]
            if 'name' in sample_w_df.columns and weight_col_candidates:
                w_col = weight_col_candidates[0]
                w_map = (
                    sample_w_df[['name', w_col]]
                    .assign(name=lambda d: d['name'].astype(str).str.strip().str.lower())
                    .drop_duplicates('name')
                    .set_index('name')[w_col]
                    .to_dict()
                )
                train_weights = train_m['name_norm'].map(lambda x: float(w_map.get(x, 1.0))).values
                sampler = WeightedRandomSampler(
                    weights=torch.tensor(train_weights, dtype=torch.double),
                    num_samples=len(train_weights),
                    replacement=True,
                )
                print(f'Weighted sampler enabled using column: {w_col}')
            else:
                print('Weighted sampler requested but sample weight file schema not compatible; fallback to shuffle.')

        train_loader = DataLoader(
            train_ds,
            batch_size=CFG['batch_size'],
            shuffle=(sampler is None),
            sampler=sampler,
            num_workers=CFG['num_workers'],
            drop_last=False,
        )
        valid_loader = DataLoader(
            valid_ds,
            batch_size=CFG['batch_size'],
            shuffle=False,
            num_workers=CFG['num_workers'],
            drop_last=False,
        )

        # optional class weights for weighted MAE
        class_weight_tensor = None
        if CFG['use_class_weights'] and class_w_df is not None:
            if {'type', 'inv_freq_weight'}.issubset(set(class_w_df.columns)):
                cw_map = class_w_df.set_index('type')['inv_freq_weight'].to_dict()
                class_weight_tensor = torch.tensor(
                    [float(cw_map.get(idx_to_class[i], 1.0)) for i in range(len(idx_to_class))],
                    dtype=torch.float32,
                    device=device,
                )
                print('Class-weighted MAE enabled.')
            else:
                print('Class weight file missing expected columns; class weighting disabled.')

        optimizer = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])

        def _weighted_mae_loss(pred, target, class_idx):
            err = torch.abs(pred - target)
            if class_weight_tensor is not None:
                err = err * class_weight_tensor[class_idx]
            return err.mean()

        best_valid_mae = float('inf')
        best_epoch = -1
        history = []
        best_pred_df = None
        best_pc_df = None

        for epoch in range(1, CFG['epochs'] + 1):
            # ---- train ----
            model.train()
            train_losses = []
            for xb, yb, cb, _meta in train_loader:
                xb = xb.to(device)
                yb = yb.to(device)
                cb = cb.to(device)

                optimizer.zero_grad()
                pred = model(xb)
                loss = _weighted_mae_loss(pred, yb, cb)
                loss.backward()
                optimizer.step()

                train_losses.append(float(loss.detach().cpu().item()))

            train_loss = float(np.mean(train_losses)) if train_losses else np.nan

            # ---- valid ----
            model.eval()
            all_pred, all_true, all_type, all_name = [], [], [], []
            valid_losses = []

            with torch.no_grad():
                for xb, yb, cb, meta in valid_loader:
                    xb = xb.to(device)
                    yb = yb.to(device)
                    cb = cb.to(device)

                    pred = model(xb)
                    loss = _weighted_mae_loss(pred, yb, cb)
                    valid_losses.append(float(loss.detach().cpu().item()))

                    all_pred.extend(pred.detach().cpu().numpy().tolist())
                    all_true.extend(yb.detach().cpu().numpy().tolist())
                    all_type.extend(meta['type'])
                    all_name.extend(meta['name'])

            valid_loss = float(np.mean(valid_losses)) if valid_losses else np.nan
            valid_mae = mae(all_true, all_pred)
            valid_rmse = rmse(all_true, all_pred)
            valid_w1 = within_1_acc(all_true, all_pred)

            eval_df = pd.DataFrame({
                'name': all_name,
                'type': all_type,
                'count': all_true,
                'pred_count': all_pred,
            })
            eval_df['abs_err'] = (eval_df['count'] - eval_df['pred_count']).abs()
            pc_df = per_class_mae(eval_df, y_pred_col='pred_count')

            history.append({
                'epoch': epoch,
                'train_loss': train_loss,
                'valid_loss': valid_loss,
                'valid_mae': valid_mae,
                'valid_rmse': valid_rmse,
                'valid_within1': valid_w1,
            })

            improved = valid_mae < best_valid_mae
            if improved:
                best_valid_mae = valid_mae
                best_epoch = epoch
                best_pred_df = eval_df.copy()
                best_pc_df = pc_df.copy()

                ckpt_path = ckpt_dir / 'best_model.pt'
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'cfg': CFG,
                    'class_to_idx': class_to_idx,
                    'best_valid_mae': best_valid_mae,
                }, ckpt_path)

            print(
                f"Epoch {epoch:02d}/{CFG['epochs']} | train_loss={train_loss:.4f} | "
                f"valid_loss={valid_loss:.4f} | valid_MAE={valid_mae:.4f} | "
                f"best_MAE={best_valid_mae:.4f} ({best_epoch})"
            )

        # ---- exports ----
        history_df = pd.DataFrame(history)
        history_path = output_dir / 'metrics_history.csv'
        history_df.to_csv(history_path, index=False)

        pred_path = output_dir / 'valid_predictions_best.csv'
        pc_path = output_dir / 'per_class_mae_best.csv'
        summary_path = output_dir / 'run_summary.json'

        if best_pred_df is not None:
            best_pred_df.to_csv(pred_path, index=False)
        if best_pc_df is not None:
            best_pc_df.to_csv(pc_path, index=False)

        summary = {
            'run_name': CFG['run_name'],
            'created_at_utc': datetime.now(timezone.utc).isoformat(),
            'best_epoch': int(best_epoch),
            'best_valid_mae': float(best_valid_mae),
            'epochs_ran': int(CFG['epochs']),
            'num_train_rows_used': int(len(train_m)),
            'num_valid_rows_used': int(len(valid_m)),
            'train_missing_features': int(train_missing),
            'valid_missing_features': int(valid_missing),
            'config': CFG,
            'artifacts': {
                'history_csv': str(history_path),
                'best_model': str(ckpt_dir / 'best_model.pt'),
                'valid_predictions_best_csv': str(pred_path),
                'per_class_mae_best_csv': str(pc_path),
            },
        }

        with open(summary_path, 'w', encoding='utf-8') as f:
            json.dump(summary, f, ensure_ascii=True, indent=2)

        TRAIN_RESULTS = {
            'history_df': history_df,
            'best_pred_df': best_pred_df,
            'best_pc_df': best_pc_df,
            'summary': summary,
        }

        print('\nSaved artifacts:')
        print(' ', history_path)
        print(' ', ckpt_dir / 'best_model.pt')
        print(' ', pred_path)
        print(' ', pc_path)
        print(' ', summary_path)



Feature index not found: ../../Data/LLSP/annotation_cleaned/pose_feature_index.csv
Create pose features first, then re-run this cell.


In [22]:
if TRAIN_RESULTS is not None:
    print('Best epoch:', TRAIN_RESULTS['summary']['best_epoch'])
    print('Best valid MAE:', TRAIN_RESULTS['summary']['best_valid_mae'])
    print('\nPer-class MAE (best):')
    display(TRAIN_RESULTS['best_pc_df'])
else:
    print('No training results in memory yet.')



No training results in memory yet.


## 8) Experiment Log
| run_id | model | pose_backbone | imbalance_strategy | valid_MAE | per_class_MAE_path | notes |
|---|---|---|---|---:|---|---|
| baseline_v1 | TemporalCountRegressor | YOLOv8/11 pose | none |  |  |  |

